# bigrams :)

In [1]:
# read the data

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

length = len(text)
print("length of the dataset - ", length)
print(text[:100])

length of the dataset -  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [2]:
# build out the vocabulary of the dataset

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)
for i in chars:
    print(i, end = '')

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

In [3]:
# create encode decode functions

stoi = {s : i for i,s in enumerate(chars)}
itos = {i : s for s,i in stoi.items()}

def encode(s):
    out = []
    for i in s:
        out.append(stoi[i])
    return out
def decode(s):
    out = ""
    for i in s:
        out+=itos[i]
    return out

# test
out = encode('hello')
print(out)
out1 = decode(out)
print(out1)

[46, 43, 50, 50, 53]
hello


In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
data[:100]

# split into train and val

limit = int(0.9 * len(data))
Xtr = data[:limit] # (torch.Size([1003854])
Xval = data[limit:] # torch.Size([1003854]))

torch.Size([1115394])


In [5]:
# setting up our contexts

torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = Xtr if split=='train' else Xval
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)

print("targets:")
print(yb.shape)

print(f"our input to the transformer {xb}")

inputs:
torch.Size([4, 8])
targets:
torch.Size([4, 8])
our input to the transformer tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [6]:
# the fun part

import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):   
        logits = self.token_embedding_table(idx) #(B,T,C)
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C) # pytorch expects this dim
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
        
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [7]:
# optimiser
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(100000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5319576263427734


In [8]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))



Ofows ht IUS:
S:

ING flvenje ssutefr,
M:
War cl igagimous pray whars:
Panalit I It aithit terised thevermenghau buaror VOubed spo mng as chathab llll:
Ware,

ee her,
Thooured aly y hindr's.
Fashat--
MNGes s, share hathure Anfaneof f s llon!

ICLiroushange

Then
Magend cugss, be jollrty

AROUFLom, ifay wil wher, gheatalloult llats howheprshakengayoref f f abighine ck orors n s?
NGABerd Foutheig vemy.
NG t isoststor hnor 'myougorme whe s'car n r toun t pridie are he of t ad
BY:
Hatamethat vint i
